In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types  import *
import sys
sys.path.append("/Workspace/Users/sivana9908_gmail.com#ext#@sivana9908gmail.onmicrosoft.com/Uber-Eats-End-to-End-_Azure-Data-Engineering-Project")
from src.common.spark_utils import standardize_columns

%-reading data from adls datalake-%

In [0]:

df_customers = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load("abfss://bronze@ubereaststorage.dfs.core.windows.net/sql/customers/")
)
display(df_customers)


# Standardize columns

In [0]:
df_customers = standardize_columns(df_customers)
display(df_customers)

# Standarize the data types

In [0]:

df_customers = df_customers.withColumn("customer_id",col("customer_id").cast("int"))\
                       .withColumn("signup_date",to_date(col("signup_date")))\
                           .withColumn("created_at",to_timestamp(col("created_at")))\
                                   .withColumn("updated_at",to_timestamp(col("updated_at")))
df_customers.printSchema()

#standardize values

In [0]:
df_customers = df_customers.withColumn("city",initcap(col("city")))\
    .withColumn("status", upper(col("status")))


#deduplication

In [0]:
df_customers=df_customers.dropDuplicates(["customer_id"])

#handling nulls

In [0]:
df_customers = df_customers.filter(col("customer_id").isNotNull())

# applying business rules

In [0]:
# status should contain only these values
df_customers = df_customers.filter(
    col("status").isin("ACTIVE", "INACTIVE")
)

# Signup date should not be NULL
df_customers = df_customers.filter(
    col("signup_date").isNotNull()
)



#validations

In [0]:
df_customers.groupBy("customer_id").count().filter(col("count")>1).show()
df_customers.filter(col("customer_id").isNull()).show()
df_customers.filter(col("status").isNull()).show()


# creating managed tables for silver cusotler

In [0]:
df_customers.write.format("delta").mode("overwrite").saveAsTable("ubereats_databricks1.silver.silver_customers")